In [1]:
%pip install -q ezkl onnx onnxruntime psutil numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, json, time, inspect, threading
from pathlib import Path

import numpy as np
import pandas as pd
import onnxruntime as ort
import psutil
import ezkl

MODEL = "linear"
CIRCUIT_SIZE = 12 #192 x 24 layers should fit in
SCALE = 13 #decial to integer just like BIT_LEN for deep prove

#paths for the files
model_dir = Path("../../fuse_models") / MODEL
all_sites = sorted(model_dir.glob("site_*"))
assert all_sites, f"no trained sites under {model_dir}, run prep/train_export.py first" #if they can't find it stop it

sites = all_sites

work_dir    = Path("_work"); work_dir.mkdir(exist_ok=True)
results_dir = Path("../../results"); results_dir.mkdir(parents=True, exist_ok=True)

def site_id(path): return path.name.replace("site_", "")

#some ezkl functions are async so if the result is a promise, it should await
async def call(fn, *args, **kwargs):
    out = fn(*args, **kwargs)
    return await out if inspect.isawaitable(out) else out

print(len(all_sites), "sites available |", len(sites), "selected")

321 sites available | 321 selected


In [ ]:
#build the circuit blue print

settings_path = str(work_dir / "settings.json")
srs_path      = str(work_dir / "kzg.srs")
sample_site   = sites[0]

#argument setup for ezkl
#the input is invisible, and the model weights are fixed and output is public
run_args = ezkl.PyRunArgs()
run_args.input_visibility  = "private"
run_args.param_visibility  = "fixed"
run_args.output_visibility = "public"
run_args.input_scale = SCALE
run_args.param_scale = SCALE

#call gen_settings function to generate the circuit settings based on the sample site.  
assert await call(ezkl.gen_settings, str(sample_site / "network.onnx"), settings_path, py_run_args=run_args)

#and then call calibrate_setting function to actually run the model on input and adjust the setting to improve the draft
await call(ezkl.calibrate_settings, str(sample_site / "input.json"), str(sample_site / "network.onnx"), settings_path, "resources")


#read the settings
settings = json.load(open(settings_path))
settings["run_args"]["logrows"] = CIRCUIT_SIZE

#save the settings
json.dump(settings, open(settings_path, "w"))
ezkl.gen_srs(srs_path, CIRCUIT_SIZE) #generates the shared setup file

print("settings ready | logrows:", settings["run_args"]["logrows"],
      "| input_scale:", settings["run_args"]["input_scale"])



 <------------- Numerical Fidelity Report (input_scale: 13, param_scale: 13, scale_input_multiplier: 10) ------------->

+-----------------+----------------+---------------+----------------+----------------+------------------+---------------+----------------+--------------------+--------------------+------------------------+
| mean_error      | median_error   | max_error     | min_error      | mean_abs_error | median_abs_error | max_abs_error | min_abs_error  | mean_squared_error | mean_percent_error | mean_abs_percent_error |
+-----------------+----------------+---------------+----------------+----------------+------------------+---------------+----------------+--------------------+--------------------+------------------------+
| -0.000008969878 | 0.000048667192 | 0.00026090443 | -0.00031533837 | 0.00013401546  | 0.000048667192   | 0.00031533837 | 0.000006765127 | 0.000000026477332  | -0.00039595482     | 0.0010920534           |
+-----------------+----------------+---------------+-

settings ready | logrows: 12 | input_scale: 13


In [ ]:
process = psutil.Process()

#helper function to calculate the memory, CPU time 
def profile(fn):
    peak = [process.memory_info().rss]
    stop = threading.Event()
    def sample():
        while not stop.is_set():
            peak[0] = max(peak[0], process.memory_info().rss)
    sampler = threading.Thread(target=sample, daemon=True); sampler.start()
    cpu_start, wall_start = process.cpu_times(), time.perf_counter()
    try:
        out = fn()
    finally:
        stop.set(); sampler.join()
    total_time = time.perf_counter() - wall_start
    cpu_end = process.cpu_times()
    cpu_time = (cpu_end.user - cpu_start.user) + (cpu_end.system - cpu_start.system)
    return out, total_time, peak[0] / 1024**2, (100 * cpu_time / total_time if total_time else None)

#set up paths for these work files 
compiled_path = str(work_dir / "network.compiled")
vk_path, pk_path = str(work_dir / "vk.key"), str(work_dir / "pk.key")
witness_path = str(work_dir / "witness.json")
proof_path   = str(work_dir / "proof.json")

rows = []

for site_dir in sites:
    #load the model and get the info 
    onnx_path, input_path = str(site_dir / "network.onnx"), str(site_dir / "input.json")
    meta = json.load(open(site_dir / "meta.json"))

    #complile the model
    assert ezkl.compile_circuit(onnx_path, compiled_path, settings_path)
    assert ezkl.setup(compiled_path, vk_path, pk_path, srs_path)

    #run the input through the circuit
    #even with the same circuit since the model has different weight, it should be run in a loop
    await call(ezkl.gen_witness, input_path, compiled_path, witness_path) 


    #make the actual prove and check it
    #lambda makes the profile function run the prove function later so that it can calculate it.
    _, prove_s, peak_mb, cpu_pct = profile(
        lambda: ezkl.prove(witness_path, compiled_path, pk_path, proof_path, srs_path=srs_path))

    #get the resources and time
    verify_start = time.perf_counter()
    verified = ezkl.verify(proof_path, settings_path, vk_path, srs_path)
    verify_s = time.perf_counter() - verify_start

    #make sure the accuracy gap of the decimal and the quantization is super small. 
    #model can run the decimal but can't prove the decimal 
    mean_err = median_err = None
    try:
        session = ort.InferenceSession(onnx_path) #load the ONNX file from the site
        window = np.array(json.load(open(input_path))["input_data"][0], np.float32).reshape([1, meta["seq_len"], 1])
        float_out = np.array(session.run(None, {session.get_inputs()[0].name: window})[0]).ravel()

        quant_out = np.array(json.load(open(proof_path))["pretty_public_inputs"]["rescaled_outputs"][0], float).ravel()
        n = min(float_out.size, quant_out.size); diff = np.abs(float_out[:n] - quant_out[:n])
        mean_err, median_err = float(diff.mean()), float(np.median(diff))

    except Exception as err:
        print(f"site {site_id(site_dir)} accuracy skipped:", err)

    
    rows.append({
        "framework": "ezkl", "model": MODEL, "site": meta["site"],
        "params": meta["params"], "logrows": CIRCUIT_SIZE,
        "prove_s": round(prove_s, 4), "verify_s": round(verify_s, 4),
        "cpu_pct": round(cpu_pct, 1) if cpu_pct else None,
        "peak_mem_mb": round(peak_mb, 1), "proof_kb": round(os.path.getsize(proof_path) / 1024, 3),
        "mean_abs_err": mean_err, "median_abs_err": median_err,
        "mse_float": meta["mse_float"], "verified": bool(verified),
    })
    print(f"site {site_id(site_dir):>3}: prove={prove_s:.3f}s verify={verify_s:.3f}s "
          f"mem={peak_mb:.0f}MB cpu={cpu_pct:.0f}% proof={rows[-1]['proof_kb']}KB verified={verified}")

In [5]:
results = pd.DataFrame(rows)
out_path = results_dir / f"ezkl_fused_{MODEL}.csv"
results.to_csv(out_path, index=False)
print("wrote", out_path, "(", len(results), "rows )")
results[["prove_s", "verify_s", "cpu_pct", "peak_mem_mb", "proof_kb",
         "mean_abs_err", "median_abs_err"]].describe().round(3)

wrote ../../results/ezkl_fused_linear.csv ( 321 rows )


,prove_s,verify_s,cpu_pct,peak_mem_mb,proof_kb
count,321.000,321.000,321.000,321.000,321.000
mean,0.859,0.031,475.414,328.067,29.790
std,0.083,0.004,13.222,10.570,0.044
min,0.740,0.023,416.300,258.400,29.672
25%,0.811,0.029,468.300,321.700,29.761
50%,0.838,0.030,477.500,331.400,29.788
75%,0.874,0.032,483.000,334.900,29.817
max,1.324,0.080,524.300,343.200,29.945
